# Creating and loading calendars

There are two main ways to work with calendars:

1. construct a `Calendar(...)` directly from holidays and nonworking weekdays
2. load a calendar with `Calendar.load(...)`

Use the constructor when your application already owns the holiday data. Use `load` when you want packaged calendars or a JSON file.

## Calendars that come with the package

The package ships with these calendar names:

- `ANBIMA`
- `B3`
- `Actual`

In [1]:
from bizdays import Calendar, list_calendars

{name: Calendar.load(name=name).name for name in ["ANBIMA", "B3", "Actual"]}

{'ANBIMA': 'ANBIMA', 'B3': 'B3', 'Actual': 'Actual'}

In [2]:
list_calendars()

{'packaged': ['ANBIMA', 'B3', 'Actual'],
 'pandas_market_calendars': {'available': True,
  'prefix': 'PMC/',
  'calendars': ['ASX',
   'BMF',
   'B3',
   'BSE',
   'NSE',
   'XNSE',
   'CFE',
   'CBOE_Futures',
   'CBOE_Equity_Options',
   'CBOE_Index_Options',
   'CME_Equity',
   'CBOT_Equity',
   'CME_Agriculture',
   'CBOT_Agriculture',
   'COMEX_Agriculture',
   'NYMEX_Agriculture',
   'CME_Rate',
   'CBOT_Rate',
   'CME_InterestRate',
   'CBOT_InterestRate',
   'CME_Bond',
   'CBOT_Bond',
   'CMEGlobex_Livestock',
   'CMEGlobex_Live_Cattle',
   'CMEGlobex_Feeder_Cattle',
   'CMEGlobex_Lean_Hog',
   'CMEGlobex_Port_Cutout',
   'CMEGlobex_Grains',
   'CMEGlobex_Oilseeds',
   'CME Globex Cryptocurrencies',
   'CME Globex Crypto',
   'CMEGlobex_EnergyAndMetals',
   'CMEGlobex_Energy',
   'CMEGlobex_CrudeAndRefined',
   'CMEGlobex_NYHarbor',
   'CMEGlobex_HO',
   'HO',
   'CMEGlobex_Crude',
   'CMEGlobex_CL',
   'CL',
   'CMEGlobex_Gas',
   'CMEGlobex_RB',
   'RB',
   'CMEGlobex_MicroC

`list_calendars()` is the programmatic discovery helper for calendar sources.

It returns a dictionary with:

- `packaged`: the curated calendars shipped with `bizdays`
- `pandas_market_calendars`: metadata about the optional PMC integration
- `exchange_calendars`: metadata about the optional exchange calendar integration
- `workalendar`: metadata about the optional workalendar integration

Each integration entry contains:

- `available`: whether the optional dependency is installed
- `prefix`: the prefix used with `Calendar.load(name="<prefix><calendar>")`
- `calendars`: the discovered calendar names when available, otherwise an empty list

In [3]:
info = list_calendars()
{
    "packaged": info["packaged"],
    "pmc_available": info["pandas_market_calendars"]["available"],
    "pmc_prefix": info["pandas_market_calendars"]["prefix"],
    "xcal_available": info["exchange_calendars"]["available"],
    "xcal_prefix": info["exchange_calendars"]["prefix"],
    "work_available": info["workalendar"]["available"],
    "work_prefix": info["workalendar"]["prefix"],
}

{'packaged': ['ANBIMA', 'B3', 'Actual'],
 'pmc_available': True,
 'pmc_prefix': 'PMC/',
 'xcal_available': True,
 'xcal_prefix': 'XCAL/',
 'work_available': True,
 'work_prefix': 'WORK/'}

`Actual` is the simplest packaged calendar: it has no holidays and no nonworking weekdays, so every day is a business day.

In [4]:
actual = Calendar.load(name="Actual")
actual.isbizday(["2024-01-01", "2024-01-06", "2024-01-07"])

masked_array(data=[True, True, True],
             mask=[False, False, False],
       fill_value=False)

## Creating a calendar from scratch with `Calendar(...)`

The constructor accepts:

- `holidays`: ISO date strings or date-like objects
- `weekdays`: names of nonworking weekdays
- `startdate` and `enddate`: optional calendar bounds
- `name`: a display name
- `financial`: whether to use financial counting rules

In [5]:
custom = Calendar(
    holidays=["2024-01-01", "2024-04-21", "2024-12-25"],
    weekdays=["Saturday", "Sunday"],
    startdate="2024-01-01",
    enddate="2024-12-31",
    name="Example",
    financial=True,
)
custom

Calendar: Example
Start: 2024-01-01
End: 2024-12-31
Weekdays: Saturday, Sunday
Holidays: 3
Financial: True

In [6]:
{
    "name": custom.name,
    "weekdays": custom.weekdays,
    "startdate": str(custom.startdate),
    "enddate": str(custom.enddate),
    "holidays": [str(day) for day in custom.holidays],
}

{'name': 'Example',
 'weekdays': ('Saturday', 'Sunday'),
 'startdate': '2024-01-01',
 'enddate': '2024-12-31',
 'holidays': ['2024-01-01', '2024-04-21', '2024-12-25']}

If you omit `startdate` and `enddate`, they are inferred from the holiday range. If you construct a calendar without holidays, a broad default range is used.

In [7]:
weekends_only = Calendar(
    weekdays=["Saturday", "Sunday"],
    name="WeekendsOnly",
)
{
    "startdate": str(weekends_only.startdate),
    "enddate": str(weekends_only.enddate),
    "isbizday": weekends_only.isbizday(["2024-01-05", "2024-01-06", "2024-01-07"]),
}

{'startdate': '1970-01-01',
 'enddate': '2071-01-01',
 'isbizday': masked_array(data=[True, False, False],
              mask=[False, False, False],
        fill_value=False)}

## JSON calendar layout

`Calendar.load(filename=...)` expects a JSON object with this layout:

- `name`: string
- `weekdays`: list of strings
- `holidays`: list of ISO date strings
- `financial`: boolean
- `start.date` and `end.date`: optional ISO date strings (`YYYY-MM-DD`) that set `startdate` and `enddate`; a bad value raises `ValueError`
- `adjust.from` and `adjust.to`: optional; only the defaults `"following"` and `"preceding"` are accepted, and other values raise `ValueError`

In [8]:
payload = {
    "name": "CustomJson",
    "weekdays": ["saturday", "sunday"],
    "holidays": ["2024-01-01", "2024-04-21", "2024-12-25"],
    "financial": True,
    "adjust.from": "following",
    "adjust.to": "preceding",
}
payload

{'name': 'CustomJson',
 'weekdays': ['saturday', 'sunday'],
 'holidays': ['2024-01-01', '2024-04-21', '2024-12-25'],
 'financial': True,
 'adjust.from': 'following',
 'adjust.to': 'preceding'}

In [9]:
import json
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmpdir:
    filename = Path(tmpdir) / "custom-calendar.json"
    filename.write_text(json.dumps(payload), encoding="utf-8")
    json_calendar = Calendar.load(filename=str(filename))

json_calendar

Calendar: CustomJson
Start: 2024-01-01
End: 2024-12-25
Weekdays: Saturday, Sunday
Holidays: 3
Financial: True

## Optional calendar provider integrations

If you install the optional `bizdays[pmc]` extra, `Calendar.load(name="PMC/<calendar>")` can load calendars from `pandas_market_calendars`.

If you install the optional `bizdays[xcal]` extra, `Calendar.load(name="XCAL/<calendar>")` can load calendars from `exchange_calendars`.

If you install the optional `bizdays[work]` extra, `Calendar.load(name="WORK/<code>")` can load calendars from `workalendar`.

These are not packaged calendars; they come from optional integrations.

In [10]:
{
    "pmc": Calendar.load(name="PMC/B3").name,
    "xcal": Calendar.load(name="XCAL/XNYS").name,
    "work": Calendar.load(name="WORK/FR").name,
}

{'pmc': 'PMC/B3', 'xcal': 'XCAL/XNYS', 'work': 'WORK/FR'}